# RetinaNet Detection
Este Notebook es parte de un proyecto que se puede encontrar [aqui](https://github.com/nel-eleven11/Proyecto2_DataScience), donde se busca diseñar una aplicación de datos para comparar e interactuar con diferentes modelos de visión por computadora. Primero, vamos a empezar instalando las librerías requeridas que incluyen 

In [1]:
import polars as pl
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import retinanet_resnet50_fpn
import torchvision.transforms as T
from PIL import Image
import os

## Pre-Procesamiento
A pesar de ya haber realizado un EDA, todavía debemos de preparar los datos en un formato soportado por YoloV8. Primero, vamos a empezar cargando los datos de nuestro dataset.

### Carga de Datos
Al trabajar dentro de Kaggle, podemos importar los datos y los outputs del Notebook de limpieza. Podemos revisar los directorios rápidamente

In [2]:
import os
print(os.listdir("/kaggle/input"))

['00-eda-and-cleaning', 'mosquito-data']


Luego, podemos setear algunas variables que nos serán de utilidad para saber dónde se encuentra la información.

In [3]:
RAW_DATA_PATH = "/kaggle/input/mosquito-data"
EDA_OUTPUT_PATH = "/kaggle/input/00-eda-and-cleaning"

print("raw:", os.listdir(RAW_DATA_PATH))
print("eda:", os.listdir(EDA_OUTPUT_PATH))

raw: ['train_images', 'sample_submission_phase1 (1).csv', 'test_images_phase1', 'test_phase1.csv', 'train.csv']
eda: ['__results__.html', 'val.csv', '__notebook__.ipynb', '__results___files', '__output__.json', 'train.csv', 'test.csv', 'custom.css']


Podemos ver por los resultados, que tenemos cargados ya los resultados de la limpieza en EDA_OUTPUT_PATH/train.csv, test.csv y val.csv respectivamente. Adicionalmente, las imágenes que utilizaremos se encuentran en RAW_DATA_PATH/train_images. Podemos cargar los datos hacia DataFrames utilizando Polars.

In [4]:
# Where the training images actually are
image_dir = os.path.join(RAW_DATA_PATH, "train_images")
print("image_dir:", image_dir)

# Optional: where to save trained models/checkpoints
MODEL_OUTPUT_DIR = "/kaggle/working/retinanet_checkpoints"
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
print("MODEL_OUTPUT_DIR:", MODEL_OUTPUT_DIR)

image_dir: /kaggle/input/mosquito-data/train_images
MODEL_OUTPUT_DIR: /kaggle/working/retinanet_checkpoints


In [5]:
train_df = pl.read_csv(os.path.join(EDA_OUTPUT_PATH, "train.csv"))
val_df   = pl.read_csv(os.path.join(EDA_OUTPUT_PATH, "val.csv"))
test_df  = pl.read_csv(os.path.join(EDA_OUTPUT_PATH, "test.csv"))

print("train shape:", train_df.shape)
print("val shape  :", val_df.shape)
print("test shape :", test_df.shape)

print("Columns:", train_df.columns)
print("Class labels:", train_df.select("class_label").unique())

train shape: (6396, 8)
val shape  : (800, 8)
test shape : (800, 8)
Columns: ['img_fName', 'img_w', 'img_h', 'bbx_xtl', 'bbx_ytl', 'bbx_xbr', 'bbx_ybr', 'class_label']
Class labels: shape: (6, 1)
┌────────────────────┐
│ class_label        │
│ ---                │
│ str                │
╞════════════════════╡
│ anopheles          │
│ albopictus         │
│ culex              │
│ culiseta           │
│ japonicus/koreicus │
│ aegypti            │
└────────────────────┘


Luego del sanity check, podemos confirmar que los datos fueron cargados exitosamente. Ahora, Yolo espera que las clases sean mappeadas de manera numérica.

In [6]:
# Map string labels -> ints starting at 1 (0 reserved for background)
classes = sorted(
    train_df.select("class_label")
            .unique()["class_label"]
            .to_list()
)
print("classes:", classes)

class_to_id = {cls: i for i, cls in enumerate(classes, start=1)}
print("class_to_id:", class_to_id)

# For RetinaNet / torchvision detection models (background + N classes)
num_classes = len(classes) + 1
print("num_classes (for RetinaNet):", num_classes)

train_df = train_df.with_columns(
    pl.col("class_label")
      .replace(class_to_id)
      .cast(pl.Int64)
      .alias("class_id")
)

val_df = val_df.with_columns(
    pl.col("class_label")
      .replace(class_to_id)
      .cast(pl.Int64)
      .alias("class_id")
)

test_df = test_df.with_columns(
    pl.col("class_label")
      .replace(class_to_id)
      .cast(pl.Int64)
      .alias("class_id")
)

train_df.head()

classes: ['aegypti', 'albopictus', 'anopheles', 'culex', 'culiseta', 'japonicus/koreicus']
class_to_id: {'aegypti': 1, 'albopictus': 2, 'anopheles': 3, 'culex': 4, 'culiseta': 5, 'japonicus/koreicus': 6}
num_classes (for RetinaNet): 7


img_fName,img_w,img_h,bbx_xtl,bbx_ytl,bbx_xbr,bbx_ybr,class_label,class_id
str,i64,i64,i64,i64,i64,i64,str,i64
"""92715872-3287-4bff-aa61-704797…",2448,3264,1301,1546,1641,2096,"""albopictus""",2
"""82df4b68-0f45-4afe-9215-48488b…",768,1024,220,58,659,808,"""albopictus""",2
"""331ad30a-7564-4478-b863-7bc760…",3456,4608,1169,2364,1586,2826,"""albopictus""",2
"""46f34803-f754-457d-bdb7-e581d3…",1152,2560,198,798,954,1351,"""albopictus""",2
"""5792dd8b-e690-4c3a-b061-a375ab…",3072,4080,1104,1030,2458,2911,"""anopheles""",3


### Carga a Datset de Torch y Transformaciones
El modelo de RetinaNet espera un tipo Dataset de Torch, por lo que debemos transformar nuestros datos ligeramente.

In [7]:
import os
import cv2

import torch
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

In [8]:
IMG_SIZE = 640

train_tf = A.Compose(
    [
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["labels"],
    ),
)

val_tf = A.Compose(
    [
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["labels"],
    ),
)

In [9]:
class RetinaDataset(Dataset):
    def __init__(self, df_pl, img_dir, transforms=None):
        """
        df_pl: Polars DataFrame with columns:
            img_fName, bbx_xtl, bbx_ytl, bbx_xbr, bbx_ybr, class_id
            (1 row per bbox)
        """
        self.df = df_pl.to_pandas()
        self.img_dir = img_dir
        self.transforms = transforms
        self.image_ids = self.df["img_fName"].unique()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        records = self.df[self.df["img_fName"] == image_id]

        img_path = os.path.join(self.img_dir, image_id)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        boxes = records[["bbx_xtl", "bbx_ytl", "bbx_xbr", "bbx_ybr"]].values.astype("float32")
        labels = records["class_id"].values.astype("int64")  # already 1..N

        if self.transforms is not None:
            transformed = self.transforms(
                image=image,
                bboxes=boxes,
                labels=labels,
            )
            image = transformed["image"]      # tensor CxHxW
            boxes = transformed["bboxes"]
            labels = transformed["labels"]

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
        }
        return image, target

In [10]:
def collate_fn(batch):
    return tuple(zip(*batch))

In [11]:
train_dataset = RetinaDataset(
    df_pl=train_df,
    img_dir=image_dir,
    transforms=train_tf,
)

val_dataset = RetinaDataset(
    df_pl=val_df,
    img_dir=image_dir,
    transforms=val_tf,
)

In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,        # bump up/down depending on VRAM
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn,
)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))

train batches: 1599
val batches: 50


In [13]:
from torchvision.models.detection import (
    retinanet_resnet50_fpn,
    RetinaNet_ResNet50_FPN_Weights,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

backbone_weights = RetinaNet_ResNet50_FPN_Weights.COCO_V1

model = retinanet_resnet50_fpn(
    weights=None,
    num_classes=num_classes,
).to(device)

device: cuda


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 212MB/s]


In [14]:
import torch

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.001,      # slower LR than 0.005
    momentum=0.9,
    weight_decay=0.0005,
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,  # decay after 15 epochs
    gamma=0.1,     # LR: 0.001 -> 0.0001
)

print("Optimizer and LR scheduler ready.")

Optimizer and LR scheduler ready.


In [15]:
import time

num_epochs = 20

history = []
start_time = time.time()

num_batches = len(train_loader)
# log about 10 times per epoch max
log_interval = max(1, num_batches // 10)

print(f"Starting training for {num_epochs} epochs "
      f"({num_batches} batches/epoch, log_interval={log_interval})")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    loss_components_sum = {}
    epoch_start = time.time()

    print(f"\n===== Epoch {epoch+1}/{num_epochs} =====")

    for batch_idx, (images, targets) in enumerate(train_loader):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_value = loss.item()
        running_loss += loss_value

        for k, v in loss_dict.items():
            loss_components_sum[k] = loss_components_sum.get(k, 0.0) + v.item()

        # periodic batch logging (visible in Kaggle logs, but not spammy)
        if (batch_idx + 1) % log_interval == 0 or (batch_idx + 1) == num_batches:
            avg_so_far = running_loss / (batch_idx + 1)
            print(
                f"  [Epoch {epoch+1}/{num_epochs}] "
                f"Batch {batch_idx+1}/{num_batches} "
                f"- batch_loss: {loss_value:.4f} "
                f"- avg_loss_so_far: {avg_so_far:.4f}"
            )

    epoch_time = time.time() - epoch_start
    epoch_loss = running_loss / num_batches
    epoch_components = {
        k: v / num_batches for k, v in loss_components_sum.items()
    }
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"- epoch_loss: {epoch_loss:.4f} "
        f"- time: {epoch_time:.1f}s "
        f"- lr: {current_lr:.6f}"
    )
    if epoch_components:
        comp_str = " | ".join(f"{k}: {v:.4f}" for k, v in epoch_components.items())
        print("  components:", comp_str)

    lr_scheduler.step()

    history.append(
        {
            "epoch": epoch + 1,
            "epoch_loss": epoch_loss,
            "epoch_time_sec": epoch_time,
            "lr": current_lr,
            "loss_components": epoch_components,
        }
    )

total_time = time.time() - start_time
print(f"\nTotal training time: {total_time/60:.2f} minutes")

Starting training for 20 epochs (1599 batches/epoch, log_interval=159)

===== Epoch 1/20 =====
  [Epoch 1/20] Batch 159/1599 - batch_loss: 1.7662 - avg_loss_so_far: 1.7524
  [Epoch 1/20] Batch 318/1599 - batch_loss: 1.7022 - avg_loss_so_far: 1.7496
  [Epoch 1/20] Batch 477/1599 - batch_loss: 1.7391 - avg_loss_so_far: 1.7231
  [Epoch 1/20] Batch 636/1599 - batch_loss: 1.6618 - avg_loss_so_far: 1.6770
  [Epoch 1/20] Batch 795/1599 - batch_loss: 1.6035 - avg_loss_so_far: 1.6737
  [Epoch 1/20] Batch 954/1599 - batch_loss: 1.6292 - avg_loss_so_far: 1.6674
  [Epoch 1/20] Batch 1113/1599 - batch_loss: 1.6757 - avg_loss_so_far: 1.6570
  [Epoch 1/20] Batch 1272/1599 - batch_loss: 1.6209 - avg_loss_so_far: 1.6397
  [Epoch 1/20] Batch 1431/1599 - batch_loss: 1.1851 - avg_loss_so_far: 1.6230
  [Epoch 1/20] Batch 1590/1599 - batch_loss: 0.8817 - avg_loss_so_far: 1.5786
  [Epoch 1/20] Batch 1599/1599 - batch_loss: 0.9893 - avg_loss_so_far: 1.5749
Epoch [1/20] - epoch_loss: 1.5749 - time: 1341.6s - l

In [16]:
import torch

torch.save(model.state_dict(), "/kaggle/working/my_model_weights.pth")
print("saved to /kaggle/working/my_model_weights.pth")

saved to /kaggle/working/my_model_weights.pth


In [17]:
import os
import json
import torch

MODEL_OUTPUT_DIR = "/kaggle/working/retinanet_artifacts"
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

run_name = "retinanet_r50_mosquito_20ep"

# 1) save model weights (state_dict-only, easiest for loading later)
weights_path = os.path.join(MODEL_OUTPUT_DIR, f"{run_name}_weights.pth")
torch.save(model.state_dict(), weights_path)
print("Saved weights to:", weights_path)

# 2) optionally, save a full checkpoint (weights + optimizer + history)
ckpt_path = os.path.join(MODEL_OUTPUT_DIR, f"{run_name}_full_ckpt.pth")
torch.save(
    {
        "epoch": num_epochs,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "config": {
            "num_classes": num_classes,
            "img_size": IMG_SIZE,
            "classes": classes,
        },
    },
    ckpt_path,
)
print("Saved full checkpoint to:", ckpt_path)

# 3) JSON metadata for your app / dashboards
meta_path = os.path.join(MODEL_OUTPUT_DIR, f"{run_name}_meta.json")
with open(meta_path, "w") as f:
    json.dump(
        {
            "run_name": run_name,
            "num_epochs": num_epochs,
            "total_train_time_sec": total_time,
            "num_classes": num_classes,
            "img_size": IMG_SIZE,
            "classes": classes,
            "history": history,
        },
        f,
        indent=2,
    )
print("Saved metadata to:", meta_path)

Saved weights to: /kaggle/working/retinanet_artifacts/retinanet_r50_mosquito_20ep_weights.pth
Saved full checkpoint to: /kaggle/working/retinanet_artifacts/retinanet_r50_mosquito_20ep_full_ckpt.pth
Saved metadata to: /kaggle/working/retinanet_artifacts/retinanet_r50_mosquito_20ep_meta.json
